In [ ]:
# doing the necessary imports
import pandas as pd
import numpy as np
from imblearn.over_sampling import RandomOverSampler
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# reading the data file
df=pd.read_csv('..\data\incomeData.csv')

In [ ]:
# Making sure that the data doesn't cintain unecessary spaces
df=df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [ ]:
# In this dataset missing values have been denoted by '?'
# we are replacing ? with NaN for them to be imputed down the line.
df.replace('?',np.NaN,inplace=True)

In [ ]:
df.isna().sum()

In [ ]:
# checking for the missing values
cols=df.columns
mc=[]
a=df.isna().sum()
for i in range(15):
    if a[i]>0:
        mc.append(cols[i])
        
    

In [ ]:
len(a)

In [ ]:
mc

In [ ]:
df.info()

In [ ]:
# As the columns which have missing values, they are only categorical, we'll use the categorical imputer
# Importing the categorical imputer
from sklearn_pandas import CategoricalImputer
imputer = CategoricalImputer()

In [ ]:
# imputing the missing values from the column

df['workclass']=imputer.fit_transform(df['workclass'])
df['occupation']=imputer.fit_transform(df['occupation'])
df['native-country']=imputer.fit_transform(df['native-country'])

In [ ]:
# The education column has a corresponding education-num column which has numerical values
df.drop(columns=['education'] ,inplace=True)

In [ ]:
# Extracting the categorical columns
cat_df = df.select_dtypes(include=['object']).copy()

In [ ]:
cat_df.columns

In [ ]:
cat_df.head()

In [ ]:
cat_df['Income'].unique()

In [ ]:
cat_df['Income'] = cat_df['Income'].map({'<=50K' : 0, '>50K' : 1})

In [ ]:
# Using the dummy encoding to encode the categorical columns to numericsl ones
for col in cat_df.drop('Income',axis=1).columns:
    x=cat_df[col].head(1)
    cat_df= pd.get_dummies(cat_df, columns=[col], prefix = [col], drop_first=True)
    

In [ ]:
cat_df.head()

In [ ]:
# extracting the numerical columns
num_df = df.select_dtypes(include=['int64']).copy()

In [ ]:
num_df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler=StandardScaler()
scaled_data=scaler.fit_transform(num_df)

In [ ]:
scaled_num_df= pd.DataFrame(data=scaled_data, columns=num_df.columns)

In [ ]:
scaled_num_df.head()

In [ ]:
# combining the Numerical and categorical dataframes to get the final dataset
final_df=pd.concat([scaled_num_df,cat_df], axis=1)

In [ ]:
final_df.head()

In [ ]:
# separating the feature and target columns
x=final_df.drop('Income',axis=1)
y=final_df['Income']


If we plot the distribution of the target column, we'd find that the peole with less than 50K annual income are more in number than the people with an annual income greaterthan 50K

In [ ]:
plt.hist(y)

Hence, the dataset is imbalanced. we need to introduce some random sampling to make it balanced.

In [ ]:
rdsmple = RandomOverSampler()
x_sampled,y_sampled  = rdsmple.fit_sample(x,y)

In [ ]:
# again plotting the target column
plt.hist(y_sampled)

As shown above, now the data looks to be balanced.

In [ ]:
# splitting the data into training and test set
from sklearn.model_selection import train_test_split
train_x,test_x,train_y,test_y=train_test_split(x_sampled,y_sampled, random_state=355 )

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import CategoricalNB
gnb = GaussianNB(priors=None, var_smoothing=0.05)

In [ ]:
y_pred = gnb.fit(train_x, train_y).predict(test_x)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
sc=accuracy_score(test_y,y_pred)
sc

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {"var_smoothing": [1e-9,0.1, 0.001, 0.5,0.05,0.01,1e-8,1e-7,1e-6,1e-10,1e-11]}

In [ ]:
grid = GridSearchCV(estimator=gnb, param_grid=param_grid, cv=5,  verbose=3)

In [ ]:
grid.fit(train_x, train_y)

In [ ]:
grid.best_estimator_

In [ ]:
from xgboost import XGBClassifier

In [ ]:
xgb=XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, criterion='gini', gamma=0,
              learning_rate=0.1, max_delta_step=0, max_depth=9,
              min_child_weight=1, missing=None, n_estimators=130, n_jobs=1,
              nthread=None, objective='binary:logistic', random_state=0,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
              silent=None, subsample=1, verbosity=1)

In [ ]:
y_pred = xgb.fit(train_x, train_y).predict(test_x)

In [ ]:
ac2=accuracy_score(test_y,y_pred)
ac2

In [ ]:
param_grid = {"n_estimators": [10, 50, 100, 130], "criterion": ['gini', 'entropy'],
                               "max_depth": range(2, 10, 1)}

            #Creating an object of the Grid Search class
grid = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5,  verbose=3,n_jobs=-1)

In [ ]:
#finding the best parameters
grid.fit(train_x, train_y)

In [ ]:
grid.best_estimator_